In [7]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
import numpy as np

df = pd.read_csv("llmdata.csv")

df = df.dropna(subset=["blended_cost_usd_per_1m"])

df["value_score"] = (
    df["intelligence_per_dollar"] +
    df["price_performance_ratio"] +
    df["speed_per_dollar"]
)

df_raw = df.copy()

top3_per_tier = (
    df.sort_values("value_score", ascending=False)
      .groupby("pricing_tier")
      .head(3)
)

print(top3_per_tier[["pricing_tier", "model_name"]])

cols = [
    "composite_benchmark",
    "aa_coding_index",
    "scicode",
    "output_tokens_per_second",
    "is_open_source",
    "release_year",
    "parameter_count",
]

target = "blended_cost_usd_per_1m"
tiers = df["pricing_tier"].dropna().unique()

results = {}

for tier in tiers:

    df_tier = df_raw[df_raw["pricing_tier"] == tier].copy()

    top3 = df_tier.nlargest(3, "value_score")

    train_data = df_tier.drop(top3.index)

    df_clean = train_data[cols + [target]].dropna(subset=[target])

    X = df_clean[cols].copy()
    y = np.log1p(df_clean[target])

    X["is_open_source"] = X["is_open_source"].astype(int)

    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X)

    X_train, X_val, y_train, y_val = train_test_split(
        X_imputed, y, test_size=0.3, random_state=42
    )

    model = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    y_val_real = np.expm1(y_val)
    y_pred_real = np.expm1(y_pred)

    mse = mean_squared_error(y_val_real, y_pred_real)
    r2 = r2_score(y_val_real, y_pred_real)

    X_top3 = top3[cols].copy()
    X_top3["is_open_source"] = X_top3["is_open_source"].astype(int)
    X_top3_imputed = imputer.transform(X_top3)

    top3_pred = np.expm1(model.predict(X_top3_imputed))

    results[tier] = {
        "mse": mse,
        "r2": r2,
        "top3_actual": top3[target].values,
        "top3_pred": top3_pred
    }

    print(f"\n{tier} tier")
    print("MSE:", mse)
    print("R²:", r2)
    print("Top 3 actual:", top3[target].values)
    print("Top 3 predicted:", top3_pred)

    pricing_tier                                         model_name
369       Budget                       Qwen3.5 0.8B (Non-reasoning)
355       Budget                           Qwen3.5 0.8B (Reasoning)
242       Budget                             Qwen3.5 2B (Reasoning)
110          Mid                                   Grok Code Fast 1
78           Mid                      Gemini 3.1 Flash-Lite Preview
127          Mid                       Gemini 2.5 Flash (Reasoning)
0        Premium                                    GPT-5.4 (xhigh)
39       Premium                                             Grok 4
4        Premium  Claude Sonnet 4.6 (Adaptive Reasoning, Max Eff...
43         Ultra                                             o3-pro
50         Ultra                          Claude 4 Opus (Reasoning)
34         Ultra                        Claude 4.1 Opus (Reasoning)
113         Free                            Apriel-v1.5-15B-Thinker
120         Free                            Apri